In [ ]:
import pandas as pd
import numpy as np
import wandb
import optuna

from src.utils.config import prices, load_env_file
from src.utils.files import read_file
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score # para las métricas de evaluación
from sklearn.metrics import recall_score, f1_score # para las métricas de evaluación
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
load_env_file()
df = read_file(prices, minio={"minio_write": True, "minio_read": True})

In [ ]:
df.info()

In [ ]:
def _PCA_matrix(col):
    '''
    La entrada es una Series de pandas con embeddings de imágenes, 
    la salida es un array de numpy con el PCA de dichos embeddings
    '''
    col_matrix = np.vstack(col.values)
    pca = PCA(n_components=10, random_state=42)
    col_pca = pca.fit_transform(col_matrix)
    return col_pca


def _transform_df_prices(df, image_col='v_clip', use_images=True):
    images_cols = ['v_clip', 'v_convnext', 'v_resnet']

    assert image_col in images_cols, "Seleccione una columna de imágenes válida"

    df_clean = df.copy()
    
    # Seleccionamos las variables de entrada útiles
    target_col = df_clean['price_range']
    erase_columns = ['id', 'name', 'price_overview']
    images_cols.remove(image_col)
    erase_columns.extend(images_cols)
    df_clean = df_clean.drop(columns=erase_columns, errors='ignore')

    if use_images:
        # Vamos a hacer un PCA de los embeddings
        # Forzamos a que todos los valores de la columna de los embeddings sean iterables
        zero_vector = np.zeros(512)
        df_clean[image_col] = df_clean[image_col].apply(
            lambda x: x if isinstance(x, (list, np.ndarray)) else zero_vector
        )
        
        imgs_pca = _PCA_matrix(df_clean[image_col])
        
        for i in range(10):
            df_clean[f'{image_col}_pca_{i}'] = imgs_pca[:, i]
    
    # Ya la columna de los embeddings originales no es necesaria
    df_clean = df_clean.drop(columns=[image_col])

    obj_cols = df_clean.select_dtypes(include=['object', 'str']).columns
    for col in obj_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    # Solo nos quedamos con columnas numéricas y rellenamos nulos
    df_clean = df_clean.select_dtypes(include=[np.number])
    df_clean = df_clean.fillna(0)

    scaler = StandardScaler()
    df_clean[df_clean.columns] = scaler.fit_transform(df_clean)

    df_clean['price_range'] = target_col

    return df_clean

In [ ]:
def objective(trial, X_t, y_t):
    '''
    Función objetivo a optimizar por Optuna.
    
    Args:
        trial: controla el intento actal
        X_t: variables de entrada de entrenamiento
        y_t: variable respuesta de entrenamiento
    '''

    # Optuna sugiere un valor para C entre 0.0001 y 10
    # C es un parámetro que controla la regularización del modelo para evitar overfitting
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    # Optuna sugiere el algoritmo interno a usar
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga', 'newton-cg'])
    
    # Elegiremos el método de regularización con ElasticNet
    # l1_ratio = 0 -> Regularización Ridge
    # l1_ratio = 1 -> Regularización Lasso
    if solver == 'saga':
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    else:
        # Los otros solo soportan Ridge
        l1_ratio = 0.0
        
    model = LogisticRegression(C=C, solver=solver, l1_ratio=l1_ratio, max_iter=1500, random_state=42)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_validate(model, X_t, y_t, cv=cv, scoring='f1_weighted', return_train_score=True)
    
    # Guardamos en el trial la precision media de entrenamiento
    trial.set_user_attr("train_f1", scores['train_score'].mean())
    
    return scores['test_score'].mean()


# Silenciamos los logs de Optuna para que no sature la pantalla
optuna.logging.set_verbosity(optuna.logging.WARNING)

combinations = [['v_clip', True], ['v_clip', False],
                ['v_resnet', True], ['v_convnext', True]]

results = []

for c in combinations:
    print(f"\n{'='*50}")
    print(f"Evaluando combinación: {c}")
    
    df_prepared = _transform_df_prices(df, image_col=c[0], use_images=c[1])

    X = df_prepared.drop(columns=['price_range'])
    y = df_prepared['price_range']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Optuna busca los mejores parámetros
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=250, n_jobs=-1)

    print(f"Mejor f1_weighted en Validación (CV): {study.best_value:.4f}")
    print(f"f1_weighted en Entrenamiento (CV): {study.best_trial.user_attrs['train_f1']:.4f}")
    print(f"Mejores hiperparámetros: {study.best_params}")
    
    best_params = dict(study.best_params)
    if 'penalty' not in best_params:
        best_params['penalty'] = 'l2'

    # Se usa ** para pasar un diccionario como argumentos
    best_model = LogisticRegression(**study.best_params, max_iter=1500, random_state=42)
    best_model.fit(X_train, y_train)
    
    final_preds = best_model.predict(X_test)
    test_f1 = accuracy_score(y_test, final_preds)
    
    results.append({
        'image_col':    c[0],
        'use_images':   c[1],
        'val_f1': study.best_value,
        'train_f1': study.best_trial.user_attrs['train_f1'],
        'test_f1':  test_f1,
        'overfit_gap':  study.best_trial.user_attrs['train_f1'] - study.best_value,
        'best_params':  study.best_params
    })

df_results = pd.DataFrame(results).sort_values('test_f1', ascending=False)
display(df_results)

In [ ]:
df_results = df_results.reset_index(drop=True)
best = df_results.iloc[0]
best_p = best['best_params']
df_best_p = pd.Series(best_p)
display(df_best_p)